# Exploring DocMind embedding-based indexing

This notebook explores the ChromaDB vector store and the result structure returned by `VectorStore.search`. It demonstrates:

- creating ingestion `TextChunk` objects;
- embedding and storing chunks in an in-memory ChromaDB collection;
- searching with a natural-language query;
- inspecting distances, normalized scores, and metadata;
- filtering results and upserting existing chunks.

The production default is the local Sentence Transformers model `all-MiniLM-L6-v2`. The first embedding operation may download the model weights. This notebook uses an ephemeral ChromaDB client, so its vectors disappear when the kernel stops.

In [ ]:
from pathlib import Path
import sys
from pprint import pprint
from uuid import uuid4

# Make the notebook work whether Jupyter was launched from the repository
# root or from the notebooks/ directory.
project_root = next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
     if (candidate / 'app').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Launch Jupyter from inside the DocMind repository')
sys.path.insert(0, str(project_root))

from app.indexing import VectorStore
from app.ingestion.text import TextChunk

## 1. Prepare representative chunks

In the full pipeline, these chunks are returned by `ingest_file(...)`. They are created manually here so the embedding experiment is reproducible and easy to inspect.

In [ ]:
chunks = [
    TextChunk(
        text="The project deadline is Friday. The team will review the release plan.",
        source="meeting-notes.md",
        chunk_id="meeting-notes:chunk-0",
        metadata={"file_type": "md", "page": "1"},
    ),
    TextChunk(
        text="La riunione del progetto è prevista per venerdì nella sala principale.",
        source="verbale-riunione.md",
        chunk_id="verbale-riunione:chunk-0",
        metadata={"file_type": "md", "page": "2"},
    ),
    TextChunk(
        text="The recipe uses flour, water, and olive oil.",
        source="cooking-notes.txt",
        chunk_id="cooking-notes:chunk-0",
        metadata={"file_type": "txt"},
    ),
]

for chunk in chunks:
    print(chunk.chunk_id, '->', chunk.text)

## 2. Create an in-memory vector store

Using no `persist_directory` creates an ephemeral ChromaDB client. A random collection name prevents this experiment from reusing vectors left by another notebook run.

In [ ]:
store = VectorStore(
    collection_name=f"notebook-{uuid4().hex}",
    embedding_model="all-MiniLM-L6-v2",
)

store.add_chunks(chunks)
print('Number of stored vectors:', store.size)

## 3. Search using semantic similarity

Unlike BM25, this search is not limited to exact term overlap. The embedding model maps semantically related text into nearby vectors. Chroma returns the nearest vectors according to the collection's cosine distance configuration.

In [ ]:
results = store.search("When is the project due?", top_k=3)
pprint(results)

In [ ]:
# Inspect the exact Python structure returned by VectorStore.search.
print('Result container type:', type(results))
print('Number of results:    ', len(results))

if results:
    first = results[0]
    print('Single result type:   ', type(first))
    print('Result keys:          ', list(first))
    print('Metadata type:        ', type(first['metadata']))
    print('Distance type:        ', type(first['distance']))
    print('Score type:           ', type(first['score']))

A result has the following conceptual structure:

```python
{
    'id': 'meeting-notes:chunk-0',
    'chunk_id': 'meeting-notes:chunk-0',
    'text': 'The project deadline is Friday...',
    'source': 'meeting-notes.md',
    'metadata': {'file_type': 'md', 'page': '1', 'source': 'meeting-notes.md'},
    'distance': 0.18,
    'score': 0.84,
}
```

The numerical values will vary with the model and Chroma version. Lower `distance` is better; the adapter exposes `score = 1 / (1 + distance)` so that larger scores are better for later hybrid fusion.

In [ ]:
for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. score={result['score']:.4f}, distance={result['distance']:.4f} | "
        f"{result['source']} | {result['text']}"
    )

## 4. Metadata filtering

Metadata is stored alongside the embedding and can be used to restrict results, for example to Markdown documents.

In [ ]:
markdown_results = store.search(
    "project meeting",
    top_k=5,
    metadata_filter={"file_type": "md"},
)
pprint(markdown_results)

## 5. Upsert behavior

Adding a chunk with an existing `chunk_id` replaces its document, embedding, and metadata. This makes re-ingestion safe when a source document changes.

In [ ]:
store.add_chunks([TextChunk(
    text="The project deadline moved to Monday.",
    source="meeting-notes.md",
    chunk_id="meeting-notes:chunk-0",
    metadata={"file_type": "md", "page": "1"},
)])

print('Number of stored vectors after upsert:', store.size)
pprint(store.search("When is the deadline now?", top_k=1))

## 6. Optional persistence

For an application index that should survive process restarts, pass a directory instead of using the ephemeral default:

```python
persistent_store = VectorStore(persist_directory='data/chroma')
```

The directory is intentionally not used in this notebook so repeated experiments remain isolated.